# Two Sigma Connect: Rental Listing Inquiries


<img src='https://www.hepsiemlak.com/emlak-yasam/wp-content/uploads/2018/06/ev-aldiktan-ne-zaman-sonra-emlak-vergisi-odenir.jpg'>


Bu çalışmada `Two Sigma Connect: Rental Listing Inquiries` yarışması için ilan özellikleri kullanılarak kullanıcı ilgisini tahmin eden bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. Accuracy değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/two-sigma-connect-rental-listing-inquiries.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.json', 'train.json.zip'])
test_member = find_zip_member(zip_members, ['test.json', 'test.json.zip'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [6]:
def read_table_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        if inner_name.endswith('.json'):
                            return pd.read_json(inner_f)
                        return pd.read_csv(inner_f)
            if member_name.endswith('.json'):
                return pd.read_json(f)
            return pd.read_csv(f)

train = read_table_from_zip(zip_path, train_member)
test = read_table_from_zip(zip_path, test_member)
sample_submission = read_table_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (49352, 15)
Test shape: (74659, 14)
Sample submission shape: (74659, 4)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,"Spacious 1 Bedroom 1 Bathroom in Williamsburg!Apartment Features:- Renovated Eat in Kitchen With Dishwasher- Renovated Bathroom- Beautiful Hardwood Floors- Lots of Sunlight- Great Closet Space- Freshly Painted- Heat and Hot Water Included- Live in Super Nearby L, J, M & G Trains !<br /><br />Contact Information:Kenneth BeakExclusive AgentC: 064-692-8838Email: kagglemanager@renthop.com, Text or Email to schedule a private viewing!<br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><br /><p><a website_redacted",145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Dishwasher, Hardwood Floors, Dogs Allowed, Cats Allowed]",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,"[https://photos.renthop.com/2/7170325_3bb5ac84a5a10227b17b273e79bd77b4.jpg, https://photos.renthop.com/2/7170325_a29a17a771ee6af213966699b05c8ea2.jpg, https://photos.renthop.com/2/7170325_149a898e8760cac1cad56e30cfe98baa.jpg, https://photos.renthop.com/2/7170325_f74a43d781bcc3c5588e61dd47de81ba.jpg, https://photos.renthop.com/2/7170325_e677d9d249ac99abe01aa5454c6e9f59.jpg, https://photos.renthop.com/2/7170325_960ea0e180bf2f15467b68b455db6172.jpg, https://photos.renthop.com/2/7170325_cbc1b8437155dbf7f5d63b3a0b5a45a3.jpg, https://photos.renthop.com/2/7170325_9a9f2adc2ce922e1d5394727efdf64bb.jpg, https://photos.renthop.com/2/7170325_aae2a39d536103eebb282775fab1c315.jpg, https://photos.renthop.com/2/7170325_cd290d0051b9f08e3482195dcbf6b5a6.jpg, https://photos.renthop.com/2/7170325_a2b599da7880eea1edd10c4b04250dc1.jpg, https://photos.renthop.com/2/7170325_6b83fa82d662bcb09733ac3a8a107113.jpg]",2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,"BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind yourself and your home in the center of it all. Steps from Grand Central Station, at the epicenter of Manhattan, The Centra combines convenience and luxury to create a perfectly balanced living experience. Offering newly renovated over sized apartment layouts.<br /><br />Full Time DoormanElevatorNewly Renovated HallwaysLaundry in BuildingOn-Site Parking Garage<br /><br />I operate with the utmost care and integrity. The client is my #1 priority. Contact me for a viewing of the great apartment, I'm more than confident we'll find a place for you to call home.Call/Text Keon: Email: If you require a move within 30 days write ""URGENT"" in the subject email or text message to be taken with high priority.<br /><br />One Month Free - net effective rent listed<p><a website_redacted",East 44th,"[Doorman, Elevator, Laundry in Building, Dishwasher, Hardwood Floors, No Fee]",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,"[https://photos.renthop.com/2/7092344_7663c19af02c46104bc4c569f7162ae0.jpg, https://photos.renthop.com/2/7092344_8287349abe511d195a7b6129bf24af0e.jpg, https://photos.renthop.com/2/7092344_e9e6a2b7aa95aa7564fe3318cadcf4e7.jpg, https://photos.renthop.com/2/7092344_d51ee4b92fd9246633f93afe6e86d8f0.jpg, https://photos.renthop.com/2/7092344_f0573fa184ca130b1b6000f2fa90511c.jpg, https://photos.renthop.com/2/7092344_b2a62f769a59a317b0a243000db46fd0.jpg]",3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,"**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**Looking for the perfect apartment in Midtown East - Sutton Place? Come check out the beautiful apartment in this prime location! **Mid 50's and 1st Ave** Elevator Building with 24-hour doorman, laundry room, and bike room!LARGE living space with King Bedroom!Beautiful large kitchen with stainless steel appliances including dishwasher! Stunning modern bathroom! Ample amount of closet space throughout entire apartment, and enough space to fit everything you need and more! Apartment i

## 4. Ön İşleme


In [7]:
# Bu bölümde hedef değişkeni düzenliyor ve temel eksik değer işlemlerini hazırlıyoruz.


In [8]:
target_col = 'interest_level'
label_map = {'low': 0, 'medium': 1, 'high': 2}
train[target_col] = train[target_col].map(label_map)

for df in [train, test]:
    df['created'] = pd.to_datetime(df['created'])
    df['num_photos'] = df['photos'].apply(len)
    df['num_features'] = df['features'].apply(len)
    df['num_description_words'] = df['description'].fillna('').apply(lambda x: len(str(x).split()))


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde fiyat, oda bilgisi ve tarih bilgisinden model özelliklerini çıkarıyoruz.


In [10]:
feature_cols = [
    'bathrooms', 'bedrooms', 'latitude', 'longitude', 'price',
    'num_photos', 'num_features', 'num_description_words'
]

for df in [train, test]:
    df['created_year'] = df['created'].dt.year
    df['created_month'] = df['created'].dt.month
    df['created_day'] = df['created'].dt.day
    df['created_hour'] = df['created'].dt.hour

feature_cols += ['created_year', 'created_month', 'created_day', 'created_hour']

train_x = train[feature_cols].copy()
train_y = train[target_col].copy()
test_x = test[feature_cols].copy()

for col in feature_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42, stratify=train_y
)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (39481, 12)
x_valid shape: (9871, 12)


## 6. Model Kurma


In [11]:
# Bu bölümde ilan ilgi seviyesi tahmini için CatBoost modelini kuruyoruz.


In [12]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=300, learning_rate=0.08, loss_function='MultiClass', random_seed=42, verbose=0)

## 7. Accuracy Değerlendirmesi


In [13]:
# Bu bölümde validation verisi üzerindeki accuracy değerini hesaplıyoruz.


In [14]:
valid_preds = model.predict(x_valid).astype(int).flatten()
acc = accuracy_score(y_valid, valid_preds)
print('Validation Accuracy:', round(acc, 5))


Validation Accuracy: 0.73468


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [16]:
test_probs = model.predict_proba(test_x)
submission = pd.DataFrame({
    'listing_id': test['listing_id'],
    'high': test_probs[:, 2],
    'medium': test_probs[:, 1],
    'low': test_probs[:, 0]
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (74659, 4)


,listing_id,high,medium,low
0,7142618,0.083080,0.442481,0.474440
1,7210040,0.384719,0.364167,0.251114
2,7174566,0.010234,0.054548,0.935218
3,7191391,0.367575,0.382403,0.250022
5,7171695,0.016164,0.165407,0.818429


In [17]:
output_path = '/content/two_sigma_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/two_sigma_submission.csv


## 9. Sonuç


Bu çalışmada Two Sigma Connect: Rental Listing Inquiries yarışması için ilan özellikleri kullanılarak kullanıcı ilgisini tahmin eden bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.73468 accuracy değeri elde etti ve test verisi için ilgi düzeyi olasılıkları üretildi.